# Topic 2: Data Skew — Detection, Salting, and AQE

> **Phase 4 → Week 7 → Topic 2**
>
> Prerequisites: Topic 1 (Shuffle Internals)

---

## What is Data Skew?

```
IDEAL (balanced):               SKEWED (real world):

Task 1: [■■■■■■■■■■] 100 rows   Task 1: [■■■■■■■■■■■■■■■■■■■■] 100,000 rows
Task 2: [■■■■■■■■■■] 100 rows   Task 2: [■] 50 rows
Task 3: [■■■■■■■■■■] 100 rows   Task 3: [■] 50 rows
Task 4: [■■■■■■■■■■] 100 rows   Task 4: [■] 50 rows
                                           ↑
Stage finishes in 1x           Stage finishes in 500x
                                (1 task blocks the whole stage!)
```

Skew happens when one key has far more rows than others. Common culprits:
- `NULL` values (all nulls hash to the same partition)
- Popular keys ("USA" has more rows than "Tuvalu")
- Bot/test accounts (user_id=1 has millions of events)
- Hot products (top SKU has 80% of all orders)

---

## Skew Symptoms in Spark UI

```
Stage Detail → Tasks tab:

Task   Duration   Input Size    Shuffle Write
─────  ─────────  ──────────    ─────────────
  0    2.1 min    4.7 GB        ← ONE task with huge input
  1    0.3 sec    2.1 MB        ← all others tiny
  2    0.2 sec    1.8 MB
  3    0.4 sec    2.3 MB

The stage can't finish until task 0 completes.
The entire cluster sits idle waiting for one task.
```

---

## Interview Cheat Sheet

**Q: What is data skew in Spark and why is it a problem?**
> Data skew means the data is unevenly distributed across partitions — one partition (task) has vastly more data than others. It causes a single task to become the bottleneck for the entire stage. All other tasks finish quickly, but the stage cannot complete until the skewed task finishes. This wastes cluster resources and dramatically increases job time.

**Q: How do you detect skew?**
> In Spark UI → Stage detail → Tasks tab: look for one task with 10x–100x more input size or duration than the median. In code: `df.groupBy("key").count().orderBy("count", ascending=False).show()` — a massive difference between max and median count indicates skew.

**Q: What is salting? How does it fix skew?**
> Salting adds a random suffix (0–N) to the skewed join/group key, splitting one huge partition into N smaller partitions that can be processed in parallel. On the other side of the join, you replicate each row N times with each suffix, so the keys still match. After joining/aggregating, you strip the salt suffix and re-aggregate. It trades N× more data for N× parallelism.

**Q: What is AQE skew join?**
> Adaptive Query Execution (Spark 3.0+) can detect skewed partitions at runtime by comparing partition sizes to the median. When it finds a skewed partition (> `skewedPartitionFactor × median` and > `skewedPartitionThresholdInBytes`), it automatically splits it into smaller sub-tasks and processes them in parallel. Unlike salting, AQE requires no code changes — just enable it.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
import time

spark = SparkSession.builder \
    .appName("Week7 - Data Skew") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready — AQE disabled so we can observe raw skew behavior")

## Part 1: Creating and Detecting Skew

In [ ]:
# Build a skewed dataset — key 'A' has 100x more rows
skewed_orders = spark.createDataFrame(
    [("A", i, float(i % 500)) for i in range(100_000)] +  # hot key
    [("B", i, float(i % 200)) for i in range(1_000)] +
    [("C", i, float(i % 200)) for i in range(800)] +
    [("D", i, float(i % 200)) for i in range(700)] +
    [("E", i, float(i % 200)) for i in range(500)],
    ["customer_id", "order_id", "amount"]
)

# Step 1: Always check key distribution BEFORE joining/grouping
print("Key distribution (always run this before joins on potentially skewed data):")
key_dist = skewed_orders.groupBy("customer_id").count() \
    .orderBy(F.col("count").desc())
key_dist.show()

# Skew ratio = max / median
stats = key_dist.agg(
    F.max("count").alias("max_count"),
    F.percentile_approx("count", 0.5).alias("median_count")
).collect()[0]

print(f"Max count:    {stats.max_count:,}")
print(f"Median count: {stats.median_count:,}")
print(f"Skew ratio:   {stats.max_count / stats.median_count:.1f}x")
print()
print("Rule: skew ratio > 10x → fix it. > 100x → definitely fix it.")

In [ ]:
# Observe skew effect: one partition gets all the data
print("Partition sizes after repartitioning by customer_id:")
partitioned = skewed_orders.repartition(8, "customer_id")

partition_sizes = partitioned.groupBy(
    F.spark_partition_id().alias("partition_id")
).count().orderBy("count", ascending=False)

partition_sizes.show()
print("One partition holds nearly all of 'A' — all 100k rows go to the same task")

## Part 2: Fix 1 — Salting (Manual Skew Fix)

In [ ]:
# The salting technique — step by step
# Problem: customer 'A' has 100k rows, others have ~1k
# Salt factor: split each key into N virtual keys

SALT_FACTOR = 10  # split skewed key into 10 sub-partitions

# --- LEFT SIDE: add a random salt suffix (0 to SALT_FACTOR-1) ---
orders_salted = skewed_orders \
    .withColumn("salt", (F.rand() * SALT_FACTOR).cast("int")) \
    .withColumn("salted_key", F.concat_ws("_", F.col("customer_id"), F.col("salt")))

print("Left side — salted keys for customer 'A':")
orders_salted.filter(F.col("customer_id") == "A") \
             .groupBy("salted_key").count() \
             .orderBy("salted_key") \
             .show()
print("'A' is now split into A_0, A_1, ... A_9 — 10 equal-ish partitions!")

In [ ]:
# --- RIGHT SIDE: replicate each customer row SALT_FACTOR times ---
# (One row for each salt value so keys can match)
customers_small = spark.createDataFrame(
    [("A", "Alice", "Premium"),
     ("B", "Bob",   "Standard"),
     ("C", "Carol", "Premium"),
     ("D", "Dave",  "Standard"),
     ("E", "Eve",   "Standard")],
    ["customer_id", "name", "tier"]
)

# Replicate by cross-joining with salt range [0, SALT_FACTOR)
salt_range = spark.range(SALT_FACTOR).withColumnRenamed("id", "salt")

customers_exploded = customers_small \
    .crossJoin(salt_range) \
    .withColumn("salted_key", F.concat_ws("_", F.col("customer_id"), F.col("salt")))

print(f"Original customers: {customers_small.count()} rows")
print(f"After salt explosion: {customers_exploded.count()} rows ({SALT_FACTOR}x larger)")
customers_exploded.select("customer_id", "name", "tier", "salted_key").show(15)

In [ ]:
# --- JOIN on the salted key ---
# Drop customer_id from customers_exploded to avoid ambiguity after join
customers_for_join = customers_exploded.drop("customer_id")

joined_salted = orders_salted.join(
    broadcast(customers_for_join),  # small table → broadcast after explosion
    on="salted_key",
    how="inner"
)

# --- AGGREGATE: strip salt and re-aggregate ---
result_salted = joined_salted \
    .groupBy("customer_id", "name", "tier") \
    .agg(
        F.sum("amount").alias("total_spend"),
        F.count("order_id").alias("order_count")
    ) \
    .orderBy("customer_id")

print("Final result after salted join:")
result_salted.show()
print()
print("Key insight: \'A\' was split across 10 tasks during the join")
print("Then merged back in the final groupBy (which is also distributed)")


In [ ]:
# Compare partition sizes: before vs after salting
print("BEFORE salting — partition sizes (after repartition by customer_id):")
skewed_orders.repartition(8, "customer_id") \
    .groupBy(F.spark_partition_id().alias("pid")).count() \
    .orderBy("count", ascending=False).show()

print("AFTER salting — partition sizes (after repartition by salted_key):")
orders_salted.repartition(8, "salted_key") \
    .groupBy(F.spark_partition_id().alias("pid")).count() \
    .orderBy("count", ascending=False).show()

## Part 3: Fix 2 — Salting for groupBy (No Join)

In [ ]:
# Salting for aggregation (no join, just groupBy)
# Pattern: two-phase aggregation
# Phase 1: groupBy salted_key → partial aggregates (distributed)
# Phase 2: groupBy original_key → final aggregates (small result)

SALT_FACTOR = 8

# Phase 1: partial aggregation on salted key
partial_agg = skewed_orders \
    .withColumn("salt", (F.rand() * SALT_FACTOR).cast("int")) \
    .withColumn("salted_key", F.concat_ws("_", F.col("customer_id"), F.col("salt"))) \
    .groupBy("salted_key", "customer_id") \
    .agg(
        F.sum("amount").alias("partial_sum"),
        F.count("*").alias("partial_count")
    )

print("Phase 1 — partial aggregates (salted):")
partial_agg.orderBy("customer_id", "salted_key").show(15)

# Phase 2: final aggregation on original key
final_agg = partial_agg \
    .groupBy("customer_id") \
    .agg(
        F.sum("partial_sum").alias("total_amount"),
        F.sum("partial_count").alias("total_orders")
    ) \
    .orderBy("customer_id")

print("Phase 2 — final result:")
final_agg.show()

## Part 4: Fix 3 — AQE Automatic Skew Join

In [ ]:
# Enable AQE with skew join handling
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# AQE detects skew at runtime by comparing partition sizes:
# skewed if size > skewedPartitionFactor × median AND size > skewedPartitionThresholdInBytes
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1mb")

print("""
AQE Skew Join Config:
─────────────────────────────────────────────────────────────
spark.sql.adaptive.enabled                           = true
spark.sql.adaptive.skewJoin.enabled                  = true
spark.sql.adaptive.skewJoin.skewedPartitionFactor    = 5
  → A partition is skewed if its size > 5× the median partition size
spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes = 256mb (default)
  → AND the partition is at least this large (don't split tiny partitions)

AQE splits skewed partitions into multiple sub-tasks:
  Skewed partition (4.7 GB) → splits into smaller sub-tasks automatically
  No code changes needed — just enable AQE!
─────────────────────────────────────────────────────────────
""")

# With AQE, run the join WITHOUT salting:
result_aqe = skewed_orders.join(
    customers_small,
    on="customer_id",
    how="inner"
).groupBy("customer_id", "name", "tier") \
 .agg(F.sum("amount").alias("total"), F.count("*").alias("cnt"))

result_aqe.show()

print("Check explain() — look for 'CustomShuffleReader' and 'SkewJoin' in the plan")
result_aqe.explain()

In [ ]:
# AQE vs Salting comparison
print("""
AQE Skew Join vs Manual Salting:
════════════════════════════════════════════════════════════════
                    AQE Skew Join           Manual Salting
──────────────────  ──────────────────────  ────────────────────
Code changes        None (config only)      Significant rewrite
Runtime detection   Yes (real data)         No (must know key)
Overhead            Splits partitions       Replicates right side
Multiple hot keys   Handles all             Must salt all manually
Right side size     No limit (SMJ)          Must fit in broadcast
Spark version       3.0+                    Any version

Decision tree:
  1. First: enable AQE (free, handles most cases automatically)
  2. If AQE not enough: salt the skewed key(s) manually
  3. If right side is small: just broadcast it (avoids skew entirely)
  4. Extreme skew: filter out skewed key, process separately, union
════════════════════════════════════════════════════════════════
""")

## Part 5: Fix 4 — Filter-and-Union (Extreme Skew)

In [ ]:
# When one key is SO dominant, even salting is complex
# Pattern: split into hot and cold paths, process separately

HOT_KEY = "A"  # known dominant key

# Split data
orders_hot  = skewed_orders.filter(F.col("customer_id") == HOT_KEY)
orders_cold = skewed_orders.filter(F.col("customer_id") != HOT_KEY)

print(f"Hot path  ('A'): {orders_hot.count():,} rows")
print(f"Cold path (rest): {orders_cold.count():,} rows")

# Process hot path: might broadcast customer lookup, use more partitions
hot_result = orders_hot \
    .repartition(20) \
    .groupBy("customer_id") \
    .agg(F.sum("amount").alias("total"), F.count("*").alias("cnt"))

# Process cold path: normal join
cold_result = orders_cold \
    .groupBy("customer_id") \
    .agg(F.sum("amount").alias("total"), F.count("*").alias("cnt"))

# Union results
final_result = hot_result.union(cold_result).orderBy("customer_id")
print("\nUnified result:")
final_result.show()

## Part 6: NULL Key Skew

In [ ]:
# NULL is a common skew source — all NULLs hash to the same partition
# Inner join on NULL keys: no match (nulls never equal nulls in SQL)
# But outer join on NULL keys: all nulls go to ONE partition!

df_with_nulls = spark.createDataFrame(
    [(None, i, 100.0) for i in range(50_000)] +  # 50k null keys!
    [("A", i, 100.0) for i in range(100)] +
    [("B", i, 100.0) for i in range(100)],
    ["key", "id", "value"]
)

print("Distribution including NULLs:")
df_with_nulls.groupBy("key").count().show()

print("""
NULL Key Skew Fix:
─────────────────────────────────────────────────────────────
Option 1: Drop nulls before join (if null rows don't contribute)
  df.filter(F.col('key').isNotNull())

Option 2: Replace null with a distinct sentinel value
  df.withColumn('key', F.coalesce(F.col('key'), F.lit('__NULL__')))
  # This distributes null rows across partitions based on hash('__NULL__')
  # (still one partition, but at least it's predictable)

Option 3: For null-inclusive outer joins, salt the null key
  df.withColumn('key',
    F.when(F.col('key').isNull(),
           F.concat(F.lit('__NULL__'), (F.rand()*10).cast('int').cast('string')))
    .otherwise(F.col('key')))
─────────────────────────────────────────────────────────────
""")

# Option 2 demo
df_fixed = df_with_nulls.withColumn(
    "key", F.coalesce(F.col("key"), F.lit("__NULL__"))
)
print("After null replacement:")
df_fixed.groupBy("key").count().show()

## Exercises

1. Read the orders CSV from `/workspace/week4/data/orders.csv`. Find which `customer_id` appears most frequently. Is there meaningful skew?
2. Write code to salt the orders dataset with `SALT_FACTOR=5` and join it with a customers table. Show the salted partition distribution.
3. What is the trade-off of using a very large salt factor (e.g., 100)? When would you choose 5 vs 50?
4. Why must the RIGHT side of a salted join be replicated `SALT_FACTOR` times? What happens if you don't replicate it?
5. A join has 10 partitions. The median partition size is 50MB. One partition is 800MB. With `skewedPartitionFactor=5` and `skewedPartitionThresholdInBytes=256mb`, will AQE split the large partition? Show your calculation.

In [ ]:
# Exercise 1: Check real data for skew
orders = spark.read.csv("/workspace/week4/data/orders.csv", header=True, inferSchema=True)
print("Orders per customer (skew check):")
orders.groupBy("customer_id").count() \
      .orderBy(F.col("count").desc()).show()

# Exercise 5: AQE skew check calculation
median = 50
factor = 5
threshold_mb = 256
large_partition_mb = 800

condition1 = large_partition_mb > factor * median
condition2 = large_partition_mb > threshold_mb

print(f"\nAQE Skew Check:")
print(f"  Condition 1: {large_partition_mb}MB > {factor}×{median}MB = {factor*median}MB? {condition1}")
print(f"  Condition 2: {large_partition_mb}MB > {threshold_mb}MB? {condition2}")
print(f"  Will AQE split it? {condition1 and condition2}")